# Chapter 9: Retrieval — Putting It to Work
## Topic 8: The "Smart Saver FD" Proof — Demonstrating RAG Actually Works

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every topic across Chapters 7-9 has built, piece by piece, an increasingly sophisticated retrieval-and-generation pipeline — but sophistication isn't the same as proven value. This topic exists to answer the most basic, most important question a stakeholder would actually ask: does all of this machinery demonstrably outperform just asking the model directly, on a case specifically designed so the answer can't be faked?
- **"Smart Saver FD" is a hypothetical, brand-new FD product name that never existed in the model's training data and never appeared in this project's original knowledge base.** This is the entire point of choosing it as the test case: a model asked about it directly has no honest way to answer correctly, because the information genuinely doesn't exist anywhere in its parameters. This is a clean, unambiguous test — unlike testing on a well-known, widely-discussed topic (where a model might answer correctly purely by chance, from general pretraining knowledge, making it impossible to tell whether retrieval actually contributed anything).
- This connects directly to two things already established elsewhere in this notebook series: Chapter 7 Topic 2 noted that dense retrieval specifically struggles with terms the embedding model has never seen during training, and Chapter 8 Topic 8 argued that fine-tuning cannot solve a knowledge-freshness problem the way RAG's simple re-ingest mechanism can — "Smart Saver FD" is the concrete, testable instance of exactly that argument, and it becomes the project's recurring benchmark case, referenced again later for Agentic RAG (Chapter 13) specifically to test the "check the catalog only if uncertain" decision.


### 2. Internal Working — Step by Step

**The proof's actual experimental design, step by step:**

1. **Invent a plausible but entirely fictional FD product** — "Smart Saver FD" — with specific, made-up terms (an interest rate, a tenure, a minimum deposit, a distinguishing feature) that could not possibly exist in the base model's training data, since the product was never real before this test was designed.
2. **Ask the model about it with no retrieval at all** — a direct, prompted-only query, exactly as if this project had never built any retrieval infrastructure. Record what the model actually says.
3. **Classify the model's ungrounded response using Chapter 8 Topic 3's faithfulness/relevance/correctness framework:** since the model has genuinely never seen this product, any confident, specific-sounding answer it produces is a hallucination by definition — there is no real information for it to be faithful to. The interesting, measurable question is not *whether* it hallucinates (it must, since no genuine information exists for it to draw on), but *how* it fails — does it confabulate a plausible-sounding but entirely fictional detail, or does it correctly recognize the gap and decline to answer?
4. **Add the actual "Smart Saver FD" product information to the knowledge base**, then re-run the exact same query through the full RAG pipeline — retrieval (Chapter 7), prompt construction (Chapter 8 Topic 1), citation (Chapter 8 Topic 2), and RAG-specific prompting (Topic 6) all engaged.
5. **Compare the two responses directly, side by side** — this comparison is the actual proof. If the ungrounded response confidently fabricated incorrect details and the RAG-grounded response correctly cited the actual, real Smart Saver FD terms, this demonstrates retrieval's value in a way no amount of describing the pipeline's sophistication could.


### 3. How This Is Implemented in This Project

- The "Smart Saver FD" test needs to be re-runnable as an actual, standing test case — not a one-time manual demonstration — so it can be used again whenever a pipeline component changes (a new embedding model, a prompt update from Topic 6, a new chunking strategy from Topic 10) to confirm the fix or change didn't regress this specific, well-understood scenario.
- This directly sets up Chapter 13's Agentic RAG topic, where the same Smart Saver FD scenario tests something more specific: not just "does RAG help when explicitly triggered," but "does the model correctly *decide* to check the knowledge base when faced with an unfamiliar term," connecting back to this notebook's own Topic 1 (retrieval triggering) and Topic 3 (wiring the tool in) — this topic's proof is the generation-quality half of that story; Chapter 13 tests the triggering-decision half.
- The knowledge base update itself (adding Smart Saver FD's actual terms) should follow the same ingestion and chunking discipline as any real knowledge base update (Chapter 5's ingestion pipeline, Topic 10's chunking quality) — this test case doubles as a working example of the full ingest-to-retrieval-to-generation loop functioning end to end on genuinely new content.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Choosing a genuinely novel test case is harder than it sounds, and the choice matters enormously for the proof's validity:** if the invented product name or its terms happen to be similar enough to real, well-known FD products that the model could plausibly guess correctly from general pattern-matching rather than genuine memorized information, the test loses its clean, unambiguous quality. The product name and its specific terms need to be distinctive enough that a correct-sounding ungrounded answer would have to be a coincidence, not a plausible guess.
- **A single successful demonstration is not the same as an ongoing guarantee:** this test proves the pipeline *can* work correctly for this specific scenario at this specific point in time — it should be re-run as a standing regression test whenever the pipeline changes, exactly as Chapter 7 Topic 9's evaluation harness and Chapter 9 Topic 7's ongoing hallucination measurement are both meant to be continuous, not one-time.
- **The proof only demonstrates value for the *retrieval-triggering-already-decided* case, not the triggering decision itself:** this topic's test explicitly invokes retrieval — it doesn't test whether the model would have correctly decided, on its own, to look up an unfamiliar term. That's a related but genuinely separate question, deliberately deferred to Chapter 13's Agentic RAG discussion, and conflating the two would overstate what this specific proof actually demonstrates.
- **Monitoring and debugging relevance:** if a future production incident involves a genuinely new product term the model handles poorly, the Smart Saver FD test methodology (invent-or-identify a term absent from training, compare ungrounded vs grounded responses) is the reusable diagnostic pattern for confirming whether the failure is a knowledge-freshness problem RAG should be solving, exactly the class of problem this test case was designed to validate against.
- **Cost:** running this specific test costs two model calls (ungrounded and grounded) plus whatever the retrieval pipeline itself costs — negligible compared to the value of having a clean, defensible, reusable proof point for stakeholder communication and regression testing.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **A single test case vs a broader suite of novel-product test cases:** one well-designed test case (Smart Saver FD) is sufficient to *demonstrate* the principle clearly and serves well as a standing regression check, but a single case is a weak foundation for measuring a *rate* — Chapter 9 Topic 7's broader, ongoing hallucination measurement across many real queries is what actually establishes how often this class of problem occurs at real production scale. This topic's single proof and Topic 7's broader measurement serve different, complementary purposes and shouldn't be confused with each other.
- **How the invented product should differ from real products:** making the invented terms too similar to real products risks an ambiguous result (the model could get lucky); making them too dissimilar from anything realistic risks testing a scenario that doesn't actually represent realistic future product launches. The right design threads this needle — genuinely novel, specific values, but plausible in structure and format compared to the project's real FD products.
- **Should this test run in isolation or alongside the broader evaluation harness (Chapter 7 Topic 9)?** running it as an isolated, hand-verified demonstration is valuable for the specific "does RAG obviously work" communication purpose, but it should also be added to any Recall@K/faithfulness evaluation set as one more labeled case, so it's automatically re-checked whenever the broader evaluation suite runs, rather than requiring a separate manual step to remember to re-verify it.


### 6. Alternatives and When to Use Each

- **A single, hand-verified novel-product demonstration (this topic's approach):** the right choice for a clear, defensible, communicable proof point — especially valuable when needing to explain RAG's value to a stakeholder who isn't going to read a full evaluation report, and as a standing regression test for pipeline changes.
- **A broader battery of multiple novel-term test cases:** worth building if a single case feels too narrow to trust, or if there's reason to believe different *kinds* of novel content (a new product, a new regulatory term, a new procedural step) might behave differently — should be considered once the single-case proof has already established the basic principle works.
- **Relying solely on the aggregate, ongoing hallucination rate (Topic 7) without any single clean demonstration case:** loses the specific communicative and diagnostic value of a designed, unambiguous test — the aggregate rate answers "how often does this happen across real traffic," while this topic's designed test answers the more fundamental "can the pipeline handle this class of problem at all, cleanly and demonstrably."


### 7. Common Mistakes and Production Failures

- Choosing a test product name or terms too similar to existing real products, allowing the ungrounded model to potentially guess correctly by coincidence or pattern-matching, weakening the proof's validity.
- Treating a single successful demonstration as a permanent guarantee, rather than re-running it as a standing regression test whenever the pipeline changes.
- Conflating this topic's proof (does retrieval, once triggered, produce a correctly grounded answer) with Chapter 13's separate question (does the model correctly decide to trigger retrieval in the first place) — these are related but genuinely different claims, and a stakeholder communication that blurs them risks overstating what's actually been demonstrated.
- Not incorporating this test case into the broader evaluation harness (Chapter 7 Topic 9), leaving it as a one-off manual demonstration that requires someone to remember to re-run it rather than being automatically re-checked.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why is a hypothetical, invented product name a better test case for proving RAG's value than a well-known, existing product?
  A: A model asked about a well-known product might answer correctly purely from general pretraining knowledge, making it impossible to tell whether retrieval actually contributed anything to a correct answer. An invented product genuinely could not exist in the model's training data, so any confident answer it gives without retrieval must be a hallucination — this makes the comparison between the ungrounded and grounded responses an unambiguous, clean test of retrieval's specific contribution.

- Q: What does this test actually prove, and what does it not prove?
  A: It proves that, once retrieval is triggered and correctly finds the relevant, newly-added content, the pipeline produces a correctly grounded, citable answer instead of a confident hallucination. It does not prove that the model would correctly decide to trigger retrieval on its own for an unfamiliar term in the first place — that's a separate question, specifically addressed later by Chapter 13's Agentic RAG discussion.

**Intermediate**

- Q: How would you design the specific details of an invented test product to make this proof as clean and defensible as possible?
  A: The invented terms need to be specific and distinctive enough that a correct-sounding ungrounded guess would have to be coincidental rather than a plausible pattern-match from similar real products, while still being structurally realistic (a plausible interest rate range, tenure, and format matching this project's actual FD products) so the test represents a realistic future scenario rather than an artificially contrived one.

- Q: This test is run once and passes. What's the risk of treating that as sufficient going forward?
  A: A single successful run only confirms the pipeline worked correctly at that specific point in time, with that specific configuration. Any later change — a new embedding model, an updated prompt, a different chunking strategy — could regress this specific capability without anyone noticing, unless the test is re-run as a standing check whenever the pipeline changes, exactly like any other regression test in a well-maintained system.

**Advanced**

- Q: A stakeholder asks for proof that the RAG investment was worthwhile. How would you use this test case to answer that, and what would you be careful to avoid claiming?
  A: Present the direct side-by-side comparison — the ungrounded response's specific fabricated details versus the grounded response's correct, citable terms for the same invented product — as a clear, concrete demonstration that retrieval genuinely changes the outcome for content the model could not otherwise know. Be careful not to claim this single case represents the system's overall hallucination rate or general reliability across real traffic — that broader claim requires Chapter 9 Topic 7's segmented, ongoing measurement across real production queries, and conflating a single clean demonstration with a comprehensive reliability claim would be overstating the evidence.

- Q: How does this topic's proof connect to Chapter 8 Topic 8's RAG-vs-fine-tuning argument?
  A: Chapter 8 Topic 8 argued that fine-tuning cannot solve a knowledge-freshness problem as directly as RAG can, since fine-tuned knowledge is exactly as static as its training data and updating it requires a full retraining cycle. This topic's Smart Saver FD test is the concrete, testable instance of that exact argument — a genuinely new product added to the knowledge base becomes correctly answerable through RAG's simple re-ingest mechanism, with no model retraining required at all, directly demonstrating the freshness advantage that Chapter 8 Topic 8 argued for in the abstract.

**Scenario-based**

- Q: You run this test and the RAG-grounded response also gets a detail wrong, despite the correct information being properly ingested into the knowledge base. Walk through your diagnosis.
  A: This is exactly the layered diagnostic discipline from Chapter 8 Topic 3 — first check whether the correct chunk was actually retrieved for this query (a retrieval problem if not), then check whether the model's generated answer actually matches what the retrieved chunk says (a faithfulness problem if the chunk was correct but the generated answer still got a detail wrong), and only after both of those check out would this point toward some other explanation, like an error in how the test product's information was originally ingested and chunked (a Topic 10 concern) rather than a retrieval or generation failure at all.

**Follow-up questions to expect:**

- "How would this test's design change if you wanted to test a genuinely new regulatory term rather than a new product?"
- "What would make you decide a single test case isn't sufficient anymore and a broader battery of novel-content tests is needed?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This is a specific instance of a general testing principle: designing a test case where the expected failure mode of the "wrong" approach is unambiguous and undeniable.** Choosing content the ungrounded model cannot possibly know, rather than content it might get right by chance, is what makes this a clean scientific comparison rather than an anecdote — the same principle (isolate the variable being tested by removing confounding explanations for success) applies broadly across engineering and experimental design, not just RAG evaluation.
- **This topic closes the loop between Chapter 7's retrieval-quality work, Chapter 8's generation-safety work, and this chapter's practical wiring work — it's the first point in this entire notebook series where all three are demonstrated working together on one concrete case**, rather than being validated separately in their own dedicated topics.
- **The "invented, definitely-novel-content" testing pattern generalizes directly to future capability additions:** whenever this project adds a genuinely new capability whose value depends on correctly handling content the base model couldn't otherwise know, this same test design (invent an unambiguous novel case, compare with-and-without the new capability) is the reusable template for proving that specific capability's value cleanly.

### 10. Quick Revision Summary (for last-minute interview prep)

> The Smart Saver FD test uses a deliberately invented, entirely fictional FD product — one that could not exist in the model's training data — to create a clean, unambiguous proof of RAG's value. Because the model genuinely cannot know anything real about this product, any confident answer given without retrieval must be a hallucination by definition, making the comparison between the ungrounded response (which should confabulate plausible-sounding but fabricated details) and the RAG-grounded response (which should correctly cite the actual, newly-ingested product terms) an unambiguous demonstration, not just an anecdote. This proves that retrieval, once triggered, produces a correctly grounded answer for content the model could never otherwise know — it does not prove the model would correctly decide to trigger retrieval on its own, which is Chapter 13's separate, later concern. This test should be re-run as a standing regression check whenever the pipeline changes, and incorporated into the broader evaluation harness (Chapter 7 Topic 9) rather than treated as a one-time manual demonstration — it's a clean, communicable proof point, not a substitute for Chapter 9 Topic 7's broader, ongoing, segmented hallucination measurement across real traffic.


### Module 1: The Test Setup — A Genuinely Novel Product

Define "Smart Saver FD" with specific, invented terms, and set up the ground truth this test will check both responses against.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- a genuinely novel, invented FD product
#
# "Smart Saver FD" and its specific terms below are ENTIRELY INVENTED
# for this test -- this product never existed in any real training
# data, and never existed in this project's original knowledge base
# before this test adds it. This is what makes the comparison clean.
# ------------------------------------------------------------------

# the GROUND TRUTH -- the real, correct (invented) facts about this
# product, which only exist because we are defining them right now
SMART_SAVER_FD_GROUND_TRUTH = {
    "product_name": "Smart Saver FD",
    "interest_rate_percent": 7.65,
    "tenure_months": 18,
    "minimum_deposit_inr": 25000,
    "distinguishing_feature": "quarterly payout with no premature withdrawal penalty in the first 6 months",
}

QUERY = "What is the interest rate and tenure for the Smart Saver FD, and does it have any special features?"

print("=" * 70)
print("MODULE 1: THE INVENTED TEST PRODUCT")
print("=" * 70)
print("Smart Saver FD -- ground truth (invented specifically for this test):")
for key, value in SMART_SAVER_FD_GROUND_TRUTH.items():
    print(f"  {key}: {value}")

print(f"\nTest query: '{QUERY}'")
print("\nThis product name and these specific figures do not exist in any")
print("real training data or in this project's original knowledge base --")
print("that is the entire point. Any confident, specific answer produced")
print("WITHOUT access to this exact information must be a hallucination.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: THE INVENTED TEST PRODUCT
Smart Saver FD -- ground truth (invented specifically for this test):
  product_name: Smart Saver FD
  interest_rate_percent: 7.65
  tenure_months: 18
  minimum_deposit_inr: 25000
  distinguishing_feature: quarterly payout with no premature withdrawal penalty in the first 6 months

Test query: 'What is the interest rate and tenure for the Smart Saver FD, and does it have any special features?'

This product name and these specific figures do not exist in any
real training data or in this project's original knowledge base --
that is the entire point. Any confident, specific answer produced
WITHOUT access to this exact information must be a hallucination.

Module 1 complete. Run Module 2 next.


### Module 2: The Ungrounded Response — What the Model Says With No Retrieval

Simulate the model answering with no access to the actual product information, honestly demonstrating that it MUST fabricate specific details since it has nothing real to draw on.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Ungrounded response -- no retrieval, no real information
#
# We cannot call a real LLM offline. We SIMULATE what an ungrounded
# model call would produce -- since NO real Smart Saver FD information
# has ever existed anywhere for a real model to have learned, this
# simulation reflects a genuine, structural certainty: an ungrounded
# response MUST be fabricated, not just "probably" fabricated.
# ------------------------------------------------------------------

def simulate_ungrounded_response(query: str) -> str:
    """SIMULATES an ungrounded model call. Since Smart Saver FD's real
    terms were only just invented in Module 1, and never existed in
    any real training data, a real model asked this question with no
    context WOULD have to either fabricate specific-sounding numbers
    (drawing on general patterns from real FD products it HAS seen)
    or decline to answer. We simulate the more common, riskier case:
    confident fabrication -- a realistic-sounding but WRONG answer."""
    return (
        "The Smart Saver FD typically offers an interest rate of around 7.0 percent "
        "for a 24-month tenure, with standard premature withdrawal penalties applying "
        "as per usual FD terms."
    )


ungrounded_answer = simulate_ungrounded_response(QUERY)

print("=" * 70)
print("MODULE 2: UNGROUNDED RESPONSE (no retrieval)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
print(f"Ungrounded answer:\n  '{ungrounded_answer}'")

# check this answer against the REAL ground truth -- using the same
# claim-level numeric check built in Chapter 8 Topic 3
import re

def extract_numbers(text: str) -> set:
    return set(re.findall(r'\d+(?:\.\d+)?', text))

ungrounded_numbers = extract_numbers(ungrounded_answer)
ground_truth_numbers = {str(SMART_SAVER_FD_GROUND_TRUTH["interest_rate_percent"]),
                         str(SMART_SAVER_FD_GROUND_TRUTH["tenure_months"])}

print(f"\nNumbers stated in ungrounded answer: {ungrounded_numbers}")
print(f"Actual correct numbers (ground truth): {ground_truth_numbers}")
matches = ungrounded_numbers & ground_truth_numbers
matches_display = matches if matches else "NONE"
print(f"Correct numbers matched: {matches_display}")

if not matches:
    print("\nCONFIRMED: the ungrounded response's specific figures (7.0 percent,")
    print("24 months) do NOT match the real Smart Saver FD terms (7.65 percent,")
    print("18 months) AT ALL -- this is not a subtle error, it is a confident,")
    print("specific-sounding, and completely WRONG fabrication -- exactly the")
    print("hallucination-by-necessity the theory describes.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: UNGROUNDED RESPONSE (no retrieval)
Query: 'What is the interest rate and tenure for the Smart Saver FD, and does it have any special features?'

Ungrounded answer:
  'The Smart Saver FD typically offers an interest rate of around 7.0 percent for a 24-month tenure, with standard premature withdrawal penalties applying as per usual FD terms.'

Numbers stated in ungrounded answer: {'7.0', '24'}
Actual correct numbers (ground truth): {'7.65', '18'}
Correct numbers matched: NONE

CONFIRMED: the ungrounded response's specific figures (7.0 percent,
24 months) do NOT match the real Smart Saver FD terms (7.65 percent,
18 months) AT ALL -- this is not a subtle error, it is a confident,
specific-sounding, and completely WRONG fabrication -- exactly the
hallucination-by-necessity the theory describes.

Module 2 complete. Run Module 3 next.


### Module 3: The Grounded Response — Full RAG Pipeline After Knowledge Base Update

Add the real Smart Saver FD chunk to the knowledge base, run REAL BM25 retrieval, then generate a grounded response and verify it against ground truth -- the actual side-by-side proof.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Grounded response -- REAL retrieval, REAL verification
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

# the knowledge base BEFORE this test -- does NOT contain Smart Saver FD
KNOWLEDGE_BASE_BEFORE = [
    {"text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "source": "01_FD_FAQ.pdf"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.", "source": "02_FD_Product_Guide.pdf"},
]

# the NEWLY INGESTED chunk -- this is what "adding the product to the
# knowledge base" actually looks like: real ingestion, no retraining
NEW_CHUNK = {
    "text": (f"The Smart Saver FD offers an interest rate of "
              f"{SMART_SAVER_FD_GROUND_TRUTH['interest_rate_percent']} percent for a "
              f"{SMART_SAVER_FD_GROUND_TRUTH['tenure_months']} month tenure, with a minimum "
              f"deposit of Rs {SMART_SAVER_FD_GROUND_TRUTH['minimum_deposit_inr']:,}. "
              f"It features {SMART_SAVER_FD_GROUND_TRUTH['distinguishing_feature']}."),
    "source": "02_FD_Product_Guide.pdf (updated)",
}

KNOWLEDGE_BASE_AFTER = KNOWLEDGE_BASE_BEFORE + [NEW_CHUNK]

print("=" * 70)
print("MODULE 3: KNOWLEDGE BASE UPDATE (real re-ingest, NO model retraining)")
print("=" * 70)
print(f"Knowledge base BEFORE: {len(KNOWLEDGE_BASE_BEFORE)} chunks (no Smart Saver FD info)")
print(f"Knowledge base AFTER:  {len(KNOWLEDGE_BASE_AFTER)} chunks (Smart Saver FD added)")
new_chunk_preview = NEW_CHUNK["text"][:80]
print(f"\nNewly ingested chunk: '{new_chunk_preview}...'")

# REAL BM25 retrieval over the UPDATED knowledge base
tokenized_corpus = [tokenize(c["text"]) for c in KNOWLEDGE_BASE_AFTER]
bm25 = BM25Okapi(tokenized_corpus)
scores = bm25.get_scores(tokenize(QUERY))
best_idx = scores.argmax()
retrieved_chunk = KNOWLEDGE_BASE_AFTER[best_idx]

print(f"\nREAL BM25 retrieval for query: '{QUERY}'")
retrieved_source = retrieved_chunk["source"]
print(f"  Top result (score={scores[best_idx]:.4f}): [{retrieved_source}]")
retrieved_preview = retrieved_chunk["text"][:80]
print(f"  '{retrieved_preview}...'")

def simulate_grounded_response(query: str, retrieved_text: str) -> str:
    """SIMULATES the generation step -- but unlike Module 2, this call
    genuinely HAS the correct information available in its context,
    so a faithful response is possible (not fabricated by necessity)."""
    return f"{retrieved_text} [Source: {retrieved_chunk['source']}]"

grounded_answer = simulate_grounded_response(QUERY, retrieved_chunk["text"])

print(f"\nGrounded answer:\n  '{grounded_answer}'")

grounded_numbers = extract_numbers(grounded_answer)
matches_grounded = ground_truth_numbers & grounded_numbers

print(f"\nNumbers stated in grounded answer: {grounded_numbers}")
print(f"Correct numbers matched: {matches_grounded}")

print("\n" + "=" * 70)
print("SIDE-BY-SIDE COMPARISON -- THE ACTUAL PROOF")
print("=" * 70)
print(f"\nUNGROUNDED (no retrieval): matched {len(matches)}/{len(ground_truth_numbers)} correct figures")
print(f"  '{ungrounded_answer[:70]}...'")
print(f"\nGROUNDED (RAG pipeline):   matched {len(matches_grounded)}/{len(ground_truth_numbers)} correct figures")
print(f"  '{grounded_answer[:70]}...'")

if len(matches_grounded) > len(matches):
    print("\nCONFIRMED: the RAG-grounded response correctly cites the REAL Smart")
    print("Saver FD terms, while the ungrounded response confidently fabricated")
    print("DIFFERENT, WRONG numbers -- a clean, unambiguous demonstration that")
    print("retrieval (once triggered) produces a genuinely more correct answer")
    print("for content the model could not otherwise know, achieved through")
    print("simple re-ingestion, with ZERO model retraining required.")

print("\nModule 3 complete. All Chapter 9 Topic 8 modules done.")
print()
print("=" * 70)
print("CHAPTER 9 TOPIC 8 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  An INVENTED, genuinely novel product is the clean test case --
  content the model structurally CANNOT know, making any correct
  ungrounded answer impossible rather than just unlikely.

  Ungrounded response MUST fabricate (or decline) -- demonstrated here
  with real claim-level number checking, matching 0 of the real figures.

  Grounded response, after simple re-ingestion (NO retraining), matches
  the real figures -- the concrete proof of RAG's knowledge-freshness
  advantage over fine-tuning (Chapter 8 Topic 8's argument, made real).

  This proves retrieval-once-triggered works correctly -- it does NOT
  prove the model would decide to trigger retrieval on its own for an
  unfamiliar term. That is Chapter 13's separate, later question.

  Re-run this test as a standing regression check whenever the
  pipeline changes -- a single pass is not a permanent guarantee.
""")


MODULE 3: KNOWLEDGE BASE UPDATE (real re-ingest, NO model retraining)
Knowledge base BEFORE: 2 chunks (no Smart Saver FD info)
Knowledge base AFTER:  3 chunks (Smart Saver FD added)

Newly ingested chunk: 'The Smart Saver FD offers an interest rate of 7.65 percent for a 18 month tenure...'

REAL BM25 retrieval for query: 'What is the interest rate and tenure for the Smart Saver FD, and does it have any special features?'
  Top result (score=2.0590): [02_FD_Product_Guide.pdf (updated)]
  'The Smart Saver FD offers an interest rate of 7.65 percent for a 18 month tenure...'

Grounded answer:
  'The Smart Saver FD offers an interest rate of 7.65 percent for a 18 month tenure, with a minimum deposit of Rs 25,000. It features quarterly payout with no premature withdrawal penalty in the first 6 months. [Source: 02_FD_Product_Guide.pdf (updated)]'

Numbers stated in grounded answer: {'7.65', '000', '18', '25', '6', '02'}
Correct numbers matched: {'7.65', '18'}

SIDE-BY-SIDE COMPARISON -- THE A